In [1]:
import os

from docling.document_converter import DocumentConverter
from docling_core.transforms.chunker import HierarchicalChunker
from hierarchical.postprocessor import ResultPostprocessor

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue,
)
from groq import Groq

/home/saquib-siddiqui/tensorvault/learning/cb-ai/code_files/cb-ai-venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
SOURCE = "https://raw.githubusercontent.com/tnahddisttud/sample-doc/refs/heads/main/AtliqAI_HR_Policies.pdf"

def load_document(source: str):
    """
    Parse a PDF using Docling.
    Returns a DoclingDocument object — not a plain string.
    """
    converter = DocumentConverter()
    result = converter.convert(source)
    ResultPostprocessor(result).process() # docling flatterns headers, the hirerachical parser maintains the flow
    return result.document

doc = load_document(SOURCE)
print(f"Document loaded: {doc.name}")

[INFO] 2026-09-06 22:14:25,628 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-06 22:14:25,630 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-09-06 22:14:25,651 [RapidOCR] download_file.py:60: File exists and is valid: /home/saquib-siddiqui/tensorvault/learning/cb-ai/code_files/cb-ai-venv/lib/python3.14/site-packages/rapidocr/models/PP-OCRv6_det_small.pth
[INFO] 2026-09-06 22:14:25,653 [RapidOCR] main.py:50: Using /home/saquib-siddiqui/tensorvault/learning/cb-ai/code_files/cb-ai-venv/lib/python3.14/site-packages/rapidocr/models/PP-OCRv6_det_small.pth
[INFO] 2026-09-06 22:14:26,006 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-06 22:14:26,008 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-09-06 22:14:26,012 [RapidOCR] download_file.py:60: File exists and is valid: /home/saquib-siddiqui/tensorvault/learning/cb-ai/code_files/cb-ai-venv/lib/python3.14/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-09

Document loaded: AtliqAI_HR_Policies


In [7]:
markdown_doc = doc.export_to_markdown()

print(markdown_doc[:1000])

## AtliqAI HR Policies

AtliqAI is committed to building a transparent, inclusive, and high-performance workplace. This document outlines the policies and guidelines that govern employment, conduct, compensation, and well-being at AtliqAI. All employees are expected to read, understand, and adhere to these policies from their first day of joining.

### Employment &amp; Onboarding

#### Offer and Joining Formalities

Upon acceptance of an offer letter, candidates must complete the joining formalities within the stipulated date mentioned in the offer. The HR team will share a prejoining checklist that includes submission of educational certificates, identity proof, address proof, previous employment documents, and a recent photograph. Failure to submit required documents within 7 working days of joining may result in withholding of the first salary disbursement.

#### Probation Period

All new employees at AtliqAI are placed on a probation period of 6 months from the date of joining. Dur

### Heirarchical Chunking

In [8]:
chunker   = HierarchicalChunker() # also need to experiment with the other chunkers like HybridChunker, SlidingWindowChunker, RecursiveChunker, etc.
doc_chunks = list(chunker.chunk(doc))

print(f"Total chunks: {len(doc_chunks)}")

# Inspect a raw DocChunk
sample = doc_chunks[2]
print(f"headings : {sample.meta.headings}")
print(f"text     : {sample.text[:200]}…")

Total chunks: 44
headings : ['AtliqAI HR Policies', 'Employment & Onboarding', 'Probation Period']
text     : All new employees at AtliqAI are placed on a probation period of 6 months from the date of joining. During this period, either party may terminate the employment with a notice period of 15 days. Perfo…


In [12]:
doc_chunks[3]

DocChunk(text='AtliqAI conducts a mandatory background verification for all new hires. This includes employment history verification for the last 5 years, educational qualification checks, criminal record screening, and reference checks from at least two previous managers. The verification is carried out by a third-party agency. Employment is contingent upon satisfactory results, and any discrepancy found may result in immediate termination.', meta=DocMeta(schema_name='docling_core.transforms.chunker.DocMeta', version='1.0.0', doc_items=[TextItem(self_ref='#/texts/8', parent=RefItem(cref='#/texts/7'), children=[], content_layer=<ContentLayer.BODY: 'body'>, meta=None, label=<DocItemLabel.TEXT: 'text'>, prov=[ProvenanceItem(page_no=1, bbox=BoundingBox(l=33.301339386, t=537.5026950409003, r=552.3731651343973, b=511.0578179539472, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 430))], source=[], comments=[], orig='AtliqAI conducts a mandatory background verification for

In [13]:
# transforming chunk by appending headings to the text of the chunk, so that the context is preserved in the embedding

def convert_chunk(doc_chunk) -> dict:
    """
    Convert a Docling DocChunk into a plain dict.

    headings   → list preserved as-is
    content    → paragraph text
    chunk_text → breadcrumb + content  (what gets embedded)
    """
    headings   = doc_chunk.meta.headings or []
    content    = doc_chunk.text.strip()
    breadcrumb = " > ".join(headings)
    chunk_text = f"{breadcrumb}\n\n{content}" if breadcrumb else content

    return {
        "headings":   headings,
        "content":    content,
        "chunk_text": chunk_text,
    }

chunks = [convert_chunk(c) for c in doc_chunks]

In [15]:
for chunk in chunks[:3]:
    print(f"headings   : {chunk['headings']}")
    print(f"content    : {chunk['content'][:500]}…")
    print(f"chunk_text : {chunk['chunk_text'][:500]}…")
    print()

headings   : ['AtliqAI HR Policies']
content    : AtliqAI is committed to building a transparent, inclusive, and high-performance workplace. This document outlines the policies and guidelines that govern employment, conduct, compensation, and well-being at AtliqAI. All employees are expected to read, understand, and adhere to these policies from their first day of joining.…
chunk_text : AtliqAI HR Policies

AtliqAI is committed to building a transparent, inclusive, and high-performance workplace. This document outlines the policies and guidelines that govern employment, conduct, compensation, and well-being at AtliqAI. All employees are expected to read, understand, and adhere to these policies from their first day of joining.…

headings   : ['AtliqAI HR Policies', 'Employment & Onboarding', 'Offer and Joining Formalities']
content    : Upon acceptance of an offer letter, candidates must complete the joining formalities within the stipulated date mentioned in the offer. The HR team wil

## Embedding

In [16]:
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBEDDING_MODEL)

chunk_texts = [c["chunk_text"] for c in chunks]

print(f"Embedding {len(chunk_texts)} chunks …")
embeddings = embedder.encode(chunk_texts, show_progress_bar=True)

print(f"Shape: {embeddings.shape}")   # → (N, 384)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2469.56it/s]


Embedding 44 chunks …


Batches: 100%|██████████| 2/2 [00:01<00:00,  1.07it/s]

Shape: (44, 384)


## Indexing

In [18]:
import os
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter,
    FieldCondition,
    MatchValue,
)

# Configuration
QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "docs"
RESET_COLLECTION = True  # Set to False to preserve existing points and schema

# Initialize remote client
client = QdrantClient(url=QDRANT_URL, timeout=10)

# 1. Health-check / verify connection to server
try:
    client.get_collections()
except Exception as exc:
    raise ConnectionError(
        f"Unable to reach Qdrant server at '{QDRANT_URL}'. "
        "Ensure the Qdrant container/service is running."
    ) from exc

# 2. Extract and validate vector dimension
try:
    DIM = embedder.get_embedding_dimension()
except AttributeError:
    # Fallback if using LangChain HuggingFaceEmbeddings / client instance
    DIM = getattr(getattr(embedder, "client", None), "get_sentence_embedding_dimension", lambda: None)()
    if DIM is None and hasattr(embedder, "embed_query"):
        DIM = len(embedder.embed_query("dimension_check"))

if not DIM:
    raise ValueError("Could not determine embedding vector dimensions from 'embedder'.")

# 3. Handle existing collection
if client.collection_exists(COLLECTION_NAME):
    if RESET_COLLECTION:
        client.delete_collection(COLLECTION_NAME)
        print(f"Deleted existing collection: '{COLLECTION_NAME}'.")
    else:
        print(f"Collection '{COLLECTION_NAME}' already exists. Skipping recreation.")

# 4. Create collection if it does not exist
if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=DIM,
            distance=Distance.COSINE,
        ),
    )
    print(f"Collection '{COLLECTION_NAME}' created with dimension {DIM}.")

Collection 'docs' created with dimension 384.


In [19]:
# Creating Points

points = [
    PointStruct(
        id=idx,
        vector=embedding.tolist(),
        payload={
            "headings":   chunk["headings"],   # stored as a JSON array
            "content":    chunk["content"],
            "chunk_text": chunk["chunk_text"],
        },
    )
    for idx, (chunk, embedding) in enumerate(zip(chunks, embeddings))
]

result = client.upsert(
    collection_name=COLLECTION_NAME,
    points=points,
    wait=True,
)
print(f"Indexed {len(points)} points — status: {result.status}")

Indexed 44 points — status: completed


In [20]:
info = client.get_collection(COLLECTION_NAME)
print(f"Points     : {info.points_count}")
print(f"Dimensions : {info.config.params.vectors.size}")

Points     : 44
Dimensions : 384


## Retrieval

In [21]:
def retrieve(
    query: str,
    top_k: int = 5
) -> list[dict]:
    """
    Embed the query and return the top-k most similar chunks.

    Args:
        query          : User's question.
        top_k          : Number of chunks to return.
        section_filter : Optional H2 heading to restrict the search scope.
    """
    query_vector = embedder.encode(query).tolist()

    hits = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
        with_payload=True,
    )

    return [{**hit.payload, "score": round(hit.score, 4)} for hit in hits.points]

In [22]:
results = retrieve("What is the paternity leave policy?", top_k=3)
for r in results:
    print(f"[{r['score']}]  {r['headings']}")
    print(f"  {r['content'][:200]}…\n")

[0.6866]  ['AtliqAI HR Policies', 'Leave Policy', 'Paternity Leave']
  Male employees and non-birthing partners are entitled to 10 working days of paid paternity leave, to be availed within 6 months of the child's birth or adoption. Paternity leave is a one-time entitlem…

[0.4824]  ['AtliqAI HR Policies', 'Leave Policy', 'Maternity Leave']
  Female employees who have completed at least 6 months of continuous service are entitled to 26 weeks of paid maternity leave for the first two live births. For the third child onwards, the entitlement…

[0.3642]  ['AtliqAI HR Policies', 'Separation & Exit', 'Full and Final Settlement']
  The full and final settlement (FnF) will be processed within 45 days of the employee's last working day. The FnF includes payment of pending salary, leave encashment of earned leaves (up to the carry-…



## Developing RAG Pipeline

In [23]:
SYSTEM_PROMPT = """You are a important HR assistant.
Answer the user's question using ONLY the context provided below.
If the context does not contain enough information, say so — do not make things up.
Always cite the section name when referencing specific information."""

In [27]:
def build_context(retrieved_chunks: list[dict]) -> str:
    parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        parts.append(f"[Source {i}]\n{chunk['content']}")
    return "\n\n---\n\n".join(parts)

In [28]:
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [29]:
from groq import Groq

groq_client = Groq()   # Reads GROQ_API_KEY from environment automatically
GROQ_MODEL  = "openai/gpt-oss-20b"

def rag(query: str, top_k: int = 5):
    """
    End-to-end RAG pipeline:
      1. Retrieve relevant chunks from Qdrant
      2. Format them as a context block
      3. Send context + query to Groq and return the answer
    """
    # Step 1 — Retrieve
    chunks = retrieve(query, top_k=top_k)
    if not chunks:
        return "No relevant content found in the document."

    # Step 2 — Build context
    context = build_context(chunks)

    # Step 3 — Generate
    user_message = f"Context:\n{context}\n\nQuestion: {query}"

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message},
        ],
        temperature=0.2,   # Low = factual;  High = creative
    )
    return response.choices[0].message.content, context

In [31]:
answer, context = rag("How many maternity leaves am I entitled to?")
print(answer)
print(f"{250*'='}")
print(f"\n\nSOURCES:\n {context}")

**Answer (Source 1)**  
- **First two live births:** 26 weeks of paid maternity leave.  
- **Third child onward:** 12 weeks of paid maternity leave.  

*Eligibility:* The employee must be female and have completed at least 6 months of continuous service. Maternity leave can start up to 8 weeks before the expected delivery date, and a medical certificate indicating the expected delivery date must be submitted at least 4 weeks in advance.


SOURCES:
 [Source 1]
Female employees who have completed at least 6 months of continuous service are entitled to 26 weeks of paid maternity leave for the first two live births. For the third child onwards, the entitlement is 12 weeks. Maternity leave can begin 8 weeks before the expected delivery date. The employee must submit a medical certificate indicating the expected date of delivery at least 4 weeks in advance.

---

[Source 2]
Male employees and non-birthing partners are entitled to 10 working days of paid paternity leave, to be availed within 